# Volatility Regime Modeling Pipeline

In [11]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

import pandas as pd

ROOT: c:\Users\ticho\Desktop\SIADS696


Configurations

In [12]:
RUN_INGESTION = False  # set True if you want to refetch/clean/merge in notebook

START_DATE = "2018-01-01"
ASSET_TICKERS = ["SPY", "QQQ", "AAPL"]
VIX_TICKERS = ["^VIX"]

RAW_DIR = ROOT / "data" / "raw"
CLEAN_DIR = ROOT / "data" / "clean"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports" / "baselines"

RAW_ASSETS_PATH = RAW_DIR / "ohlcv_SPY-QQQ-AAPL_2018-01-01_latest_1d.parquet"
RAW_VIX_PATH = RAW_DIR / "vix_VIX_2018-01-01_latest_1d.parquet"

CLEAN_ASSETS_PATH = CLEAN_DIR / "assets_clean.parquet"
CLEAN_VIX_PATH = CLEAN_DIR / "vix_clean.parquet"

MERGED_PATH = INTERIM_DIR / "ohlcv_with_vix.parquet"
FEATURES_PATH = PROCESSED_DIR / "vol_features_5d.parquet"

HORIZON_DAYS = 5
TRAIN_END = "2022-12-30"
VAL_START = "2023-01-03"
VAL_END = "2023-12-29"
TEST_START = "2024-01-02"
N_REGIMES = 3

Data Ingestion

In [13]:
from src.pipeline.ingest import FetchSpec, fetch_ohlcv, clean_ohlcv, merge_assets_with_vix

if RUN_INGESTION:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)

    # Fetch raw
    assets_raw = fetch_ohlcv(FetchSpec(tickers=ASSET_TICKERS, start=START_DATE))
    vix_raw = fetch_ohlcv(FetchSpec(tickers=VIX_TICKERS, start=START_DATE))

    assets_raw.to_parquet(RAW_ASSETS_PATH, index=False)
    vix_raw.to_parquet(RAW_VIX_PATH, index=False)

    # Clean
    assets_clean = clean_ohlcv(assets_raw)
    vix_clean = clean_ohlcv(vix_raw)

    assets_clean.to_parquet(CLEAN_ASSETS_PATH, index=False)
    vix_clean.to_parquet(CLEAN_VIX_PATH, index=False)

    # Merge
    merged = merge_assets_with_vix(assets_clean, vix_clean)
    merged.to_parquet(MERGED_PATH, index=False)

    print("Ingestion complete:", MERGED_PATH)

else:
    print("Skipping ingestion. Expecting merged file at:", MERGED_PATH)

Skipping ingestion. Expecting merged file at: c:\Users\ticho\Desktop\SIADS696\data\interim\ohlcv_with_vix.parquet


Reading Data

In [14]:
df = pd.read_parquet(MERGED_PATH)
df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)

print("Shape:", df.shape)
print("Date range:", df["date"].min().date(), "→", df["date"].max().date())
print("Tickers:", sorted(df["ticker"].astype(str).unique()))
df.head()

Shape: (6123, 13)
Date range: 2018-01-02 → 2026-02-13
Tickers: ['AAPL', 'QQQ', 'SPY']


,date,ticker,open,high,low,close,adj_close,volume,vix_close,vix_volume,vix_log_return,vix_rv_10,vix_rv_20
0,2018-01-02,AAPL,42.540001,43.075001,42.314999,43.064999,40.304176,102223600,9.77,0,NaN,NaN,NaN
1,2018-01-03,AAPL,43.132500,43.637501,42.990002,43.057499,40.297157,118071600,9.15,0,-0.065563,NaN,NaN
2,2018-01-04,AAPL,43.134998,43.367500,43.020000,43.257500,40.484325,89738400,9.22,0,0.007621,NaN,NaN
3,2018-01-05,AAPL,43.360001,43.842499,43.262501,43.750000,40.945263,94640000,9.22,0,0.000000,NaN,NaN
4,2018-01-08,AAPL,43.587502,43.902500,43.482498,43.587502,40.793182,82271200,9.52,0,0.032020,NaN,NaN


Feature Engineering

In [15]:
from src.pipeline.features import build_features_and_labels

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

feat = build_features_and_labels(
    merged_df=df,
    horizon_days=HORIZON_DAYS,
    train_end=TRAIN_END,
    n_regimes=N_REGIMES,
)

feat.to_parquet(FEATURES_PATH, index=False)

print("Saved features:", FEATURES_PATH)
print("Shape:", feat.shape)
feat[["date", "ticker", "regime", f"fwd_rv_{HORIZON_DAYS}"]].head()

Saved features: c:\Users\ticho\Desktop\SIADS696\data\processed\vol_features_5d.parquet
Shape: (5928, 30)


,date,ticker,regime,fwd_rv_5
0,2018-03-29,AAPL,2,0.277560
1,2018-04-02,AAPL,2,0.274886
2,2018-04-03,AAPL,2,0.292819
3,2018-04-04,AAPL,2,0.272819
4,2018-04-05,AAPL,2,0.277536


In [16]:
display(feat["regime"].value_counts().sort_index())
display(feat.groupby("ticker")["regime"].value_counts().sort_index())

regime
0    2241
1    2137
2    1550
Name: count, dtype: int64

ticker  regime
AAPL    0          433
        1          743
        2          800
QQQ     0          691
        1          771
        2          514
SPY     0         1117
        1          623
        2          236
Name: count, dtype: int64

Basline Model Training

In [17]:
from src.pipeline.train import SplitSpec, train_baselines_3way

split = SplitSpec(
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
)

res = train_baselines_3way(df=feat, target="regime", split=split)

print("=== Split ===")
print(res["split"])
print("\n#features:", len(res["features"]))

for model_name, m in res["models"].items():
    print(f"\n=== {model_name} ===")
    print("VAL  | acc:", round(m["val"]["accuracy"], 4), "macro_f1:", round(m["val"]["macro_f1"], 4))
    print("TEST | acc:", round(m["test"]["accuracy"], 4), "macro_f1:", round(m["test"]["macro_f1"], 4))
    print("VAL confusion:", m["val"]["confusion_matrix"])
    print("TEST confusion:", m["test"]["confusion_matrix"])

=== Split ===
{'train': {'start': '2018-03-29', 'end': '2022-12-30', 'n': 3597}, 'val': {'start': '2023-01-03', 'end': '2023-12-29', 'n': 750}, 'test': {'start': '2024-01-02', 'end': '2026-02-06', 'n': 1581}}

#features: 26

=== logreg ===
VAL  | acc: 0.5293 macro_f1: 0.4292
TEST | acc: 0.5123 macro_f1: 0.4644
VAL confusion: [[194, 113, 6], [133, 193, 44], [15, 42, 10]]
TEST confusion: [[455, 265, 9], [244, 291, 33], [57, 163, 64]]

=== random_forest ===
VAL  | acc: 0.5173 macro_f1: 0.424
TEST | acc: 0.5225 macro_f1: 0.4653
VAL confusion: [[230, 75, 8], [185, 146, 39], [20, 35, 12]]
TEST confusion: [[474, 221, 34], [227, 288, 53], [68, 152, 64]]


Random Forest Tuning

In [18]:
from src.pipeline.train import SplitSpec, tune_random_forest

split = SplitSpec(
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
)

tuning_df, best_params = tune_random_forest(df=feat, target="regime", split=split)

print("Best params:", best_params)
tuning_df.head(15)

Best params: {'n_estimators': 400, 'random_state': 42, 'n_jobs': -1, 'class_weight': 'balanced_subsample', 'max_depth': nan, 'min_samples_leaf': 1, 'max_features': 0.5}


,val_macro_f1,val_accuracy,max_depth,min_samples_leaf,max_features
0,0.429829,0.536000,NaN,1,0.5
1,0.423968,0.517333,NaN,1,sqrt
2,0.395332,0.517333,NaN,1,None
3,0.395324,0.520000,NaN,5,sqrt
4,0.392378,0.520000,8.0,20,None
5,0.392043,0.517333,5.0,5,sqrt
6,0.389403,0.516000,NaN,20,None
7,0.389098,0.526667,8.0,5,sqrt
8,0.388857,0.526667,8.0,1,sqrt
9,0.387653,0.512000,5.0,1,sqrt


Feature upgrade: add lagged return and lagged VIX-return features

In [19]:
# Rebuild features/labels with new lag features
from src.pipeline.features import build_features_and_labels
from src.pipeline.train import SplitSpec, train_baselines_3way

# Re-run feature construction
feat = build_features_and_labels(
    merged_df=df,
    horizon_days=HORIZON_DAYS,
    train_end=TRAIN_END,
    n_regimes=N_REGIMES,
)

# Save (optional but recommended)
FEATURES_PATH = PROCESSED_DIR / "vol_features_5d_lags.parquet"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
feat.to_parquet(FEATURES_PATH, index=False)

print("Saved:", FEATURES_PATH)
print("New shape:", feat.shape)

# Train/val/test evaluation (same split as before)
split = SplitSpec(
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
)

res = train_baselines_3way(df=feat, target="regime", split=split)

print("=== Split ===")
print(res["split"])
print("\n#features:", len(res["features"]))

for model_name, m in res["models"].items():
    print(f"\n=== {model_name} ===")
    print("VAL  | acc:", round(m["val"]["accuracy"], 4), "macro_f1:", round(m["val"]["macro_f1"], 4))
    print("TEST | acc:", round(m["test"]["accuracy"], 4), "macro_f1:", round(m["test"]["macro_f1"], 4))

Saved: c:\Users\ticho\Desktop\SIADS696\data\processed\vol_features_5d_lags.parquet
New shape: (5928, 30)
=== Split ===
{'train': {'start': '2018-03-29', 'end': '2022-12-30', 'n': 3597}, 'val': {'start': '2023-01-03', 'end': '2023-12-29', 'n': 750}, 'test': {'start': '2024-01-02', 'end': '2026-02-06', 'n': 1581}}

#features: 26

=== logreg ===
VAL  | acc: 0.5293 macro_f1: 0.4292
TEST | acc: 0.5123 macro_f1: 0.4644

=== random_forest ===
VAL  | acc: 0.5173 macro_f1: 0.424
TEST | acc: 0.5225 macro_f1: 0.4653


Logistic Regression Tuning

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.pipeline.train import SplitSpec, time_split_3way, select_numeric_features, _make_preprocessor

split = SplitSpec(
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
)

# Split
train_df, val_df, test_df = time_split_3way(feat, split)

# Features (same selection logic as your pipeline)
features = select_numeric_features(feat, target="regime")

X_train, y_train = train_df[features], train_df["regime"].astype(int)
X_val, y_val = val_df[features], val_df["regime"].astype(int)
X_test, y_test = test_df[features], test_df["regime"].astype(int)

preprocessor = _make_preprocessor(features)

grid_C = [0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0]
grid_cw = [None, "balanced"]

rows = []
best = None
best_score = -1

for C in grid_C:
    for cw in grid_cw:
        model = Pipeline([
            ("prep", preprocessor),
            ("clf", LogisticRegression(max_iter=4000, C=C, class_weight=cw)),
        ])

        model.fit(X_train, y_train)
        pred_val = model.predict(X_val)

        val_f1 = float(f1_score(y_val, pred_val, average="macro"))
        val_acc = float(accuracy_score(y_val, pred_val))

        rows.append({
            "C": C,
            "class_weight": cw,
            "val_macro_f1": val_f1,
            "val_accuracy": val_acc,
        })

        if val_f1 > best_score:
            best_score = val_f1
            best = (C, cw)

tune_df = pd.DataFrame(rows).sort_values(["val_macro_f1", "val_accuracy"], ascending=False).reset_index(drop=True)
display(tune_df.head(15))

best_C, best_cw = best
print("Best (by VAL macro-F1):", {"C": best_C, "class_weight": best_cw, "val_macro_f1": best_score})

# Final: retrain on train+val, test once
trainval = pd.concat([train_df, val_df], ignore_index=True)
X_trainval, y_trainval = trainval[features], trainval["regime"].astype(int)

final_model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=4000, C=best_C, class_weight=best_cw)),
])
final_model.fit(X_trainval, y_trainval)

pred_test = final_model.predict(X_test)
test_f1 = float(f1_score(y_test, pred_test, average="macro"))
test_acc = float(accuracy_score(y_test, pred_test))

print("Final TEST:", {"accuracy": round(test_acc, 4), "macro_f1": round(test_f1, 4)})

,C,class_weight,val_macro_f1,val_accuracy
0,1.00,NaN,0.429150,0.529333
1,1.00,balanced,0.429150,0.529333
2,10.00,NaN,0.420226,0.508000
3,10.00,balanced,0.420226,0.508000
4,3.00,NaN,0.418404,0.512000
5,3.00,balanced,0.418404,0.512000
6,100.00,NaN,0.417339,0.506667
7,100.00,balanced,0.417339,0.506667
8,30.00,NaN,0.415812,0.504000
9,30.00,balanced,0.415812,0.504000


Best (by VAL macro-F1): {'C': 1.0, 'class_weight': None, 'val_macro_f1': 0.4291503946662943}
Final TEST: {'accuracy': 0.5123, 'macro_f1': 0.4504}


Gradient Boosting

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline

from src.pipeline.train import SplitSpec, time_split_3way, select_numeric_features, _make_preprocessor

# Same split as before
split = SplitSpec(
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
)

train_df, val_df, test_df = time_split_3way(feat, split)

features = select_numeric_features(feat, target="regime")

X_train, y_train = train_df[features], train_df["regime"].astype(int)
X_val, y_val = val_df[features], val_df["regime"].astype(int)
X_test, y_test = test_df[features], test_df["regime"].astype(int)

preprocessor = _make_preprocessor(features)

# Simple, reasonable starting configuration
gb_model = Pipeline([
    ("prep", preprocessor),
    ("clf", HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_depth=None,
        random_state=42
    )),
])

# Train on TRAIN only
gb_model.fit(X_train, y_train)

# Validation evaluation
val_pred = gb_model.predict(X_val)
val_f1 = f1_score(y_val, val_pred, average="macro")
val_acc = accuracy_score(y_val, val_pred)

print("Gradient Boosting — VAL")
print("Accuracy:", round(val_acc, 4))
print("Macro F1:", round(val_f1, 4))

Gradient Boosting — VAL
Accuracy: 0.5107
Macro F1: 0.4194
